# Saliency Maps for Trading

This notebook demonstrates how to use saliency maps to understand and improve neural network-based trading models.

## Contents
1. Setup and Data Loading
2. Feature Engineering
3. Model Training
4. Saliency Map Computation
5. Visualization and Interpretation
6. Saliency-Based Trading Strategy
7. Backtesting and Comparison

In [ ]:
# Standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# PyTorch
import torch
import torch.nn as nn

# Local modules
import sys
sys.path.append('.')

from python.model import TradingLSTM, create_model
from python.saliency import (
    SaliencyComputer,
    aggregate_temporal_importance,
    aggregate_feature_importance,
    compute_saliency_concentration,
    plot_saliency_heatmap,
    plot_temporal_importance,
    plot_feature_importance
)
from python.train import (
    fetch_stock_data,
    compute_technical_indicators,
    prepare_sequences,
    create_dataloaders,
    train_model,
    evaluate_model
)
from python.backtest import (
    SaliencyTradingStrategy,
    run_backtest,
    compare_strategies
)

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

## 1. Setup and Data Loading

We'll use Yahoo Finance to fetch historical stock data for our analysis.

In [ ]:
# Configuration
SYMBOL = 'AAPL'
START_DATE = '2018-01-01'
END_DATE = '2023-12-31'
SEQUENCE_LENGTH = 30  # Number of days to look back
BATCH_SIZE = 32
HIDDEN_SIZE = 64
NUM_LAYERS = 2
EPOCHS = 50
LEARNING_RATE = 0.001

# Fetch data
print(f"Fetching data for {SYMBOL}...")
df = fetch_stock_data(SYMBOL, START_DATE, END_DATE)
print(f"Data shape: {df.shape}")
df.head()

In [ ]:
# Plot price history
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(df.index, df['Close'], label='Close Price')
axes[0].set_ylabel('Price ($)')
axes[0].set_title(f'{SYMBOL} Price History')
axes[0].legend()

axes[1].bar(df.index, df['Volume'], alpha=0.7, label='Volume')
axes[1].set_ylabel('Volume')
axes[1].set_xlabel('Date')
axes[1].legend()

plt.tight_layout()
plt.show()

## 2. Feature Engineering

We'll compute various technical indicators to use as features for our model.

In [ ]:
# Compute technical indicators
df_features = compute_technical_indicators(df)

# Define feature columns
FEATURE_COLUMNS = [
    'Open', 'High', 'Low', 'Close', 'Volume',
    'Returns', 'SMA_5', 'SMA_20', 'MACD', 'RSI',
    'BB_Width', 'Volatility', 'Volume_Ratio'
]

print(f"Features: {FEATURE_COLUMNS}")
print(f"\nFeature statistics:")
df_features[FEATURE_COLUMNS].describe()

In [ ]:
# Prepare sequences for training
X, y = prepare_sequences(
    df_features,
    sequence_length=SEQUENCE_LENGTH,
    feature_columns=FEATURE_COLUMNS
)

print(f"Features shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"Class distribution: {np.bincount(y.astype(int))}")

In [ ]:
# Create data loaders
train_loader, val_loader, test_loader = create_dataloaders(
    X, y,
    batch_size=BATCH_SIZE,
    test_size=0.2,
    val_size=0.1
)

print(f"Training samples: {len(train_loader.dataset)}")
print(f"Validation samples: {len(val_loader.dataset)}")
print(f"Test samples: {len(test_loader.dataset)}")

## 3. Model Training

We'll train an LSTM model to predict price direction.

In [ ]:
# Create model
model = create_model(
    model_type='lstm',
    input_size=len(FEATURE_COLUMNS),
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    dropout=0.2
)

print(model)

In [ ]:
# Train model
history = train_model(
    model,
    train_loader,
    val_loader,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    early_stopping_patience=10
)

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train Loss')
axes[0].plot(history['val_loss'], label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()

# Accuracy
axes[1].plot(history['train_acc'], label='Train Acc')
axes[1].plot(history['val_acc'], label='Val Acc')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training and Validation Accuracy')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Evaluate on test set
metrics = evaluate_model(model, test_loader)
print("Test Set Metrics:")
for name, value in metrics.items():
    print(f"  {name}: {value:.4f}")

## 4. Saliency Map Computation

Now we'll compute saliency maps to understand what the model focuses on.

In [ ]:
# Create saliency computer
saliency_computer = SaliencyComputer(model)

# Get a batch of test samples
test_features, test_labels = next(iter(test_loader))
sample_input = test_features[:1]  # Single sample

print(f"Sample input shape: {sample_input.shape}")

In [ ]:
# Compute saliency maps using different methods
saliency_maps = saliency_computer.compute_all(sample_input)

print("Computed saliency maps:")
for method, saliency in saliency_maps.items():
    print(f"  {method}: shape {saliency.shape}")

## 5. Visualization and Interpretation

Let's visualize the saliency maps to understand what the model is focusing on.

In [ ]:
# Compare saliency methods
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

methods = ['vanilla', 'gradient_x_input', 'integrated', 'smoothgrad']
titles = ['Vanilla Gradient', 'Gradient x Input', 'Integrated Gradients', 'SmoothGrad']

for ax, method, title in zip(axes.flatten(), methods, titles):
    saliency = saliency_maps[method][0].numpy()
    im = ax.imshow(saliency.T, aspect='auto', cmap='hot')
    ax.set_xlabel('Time Step')
    ax.set_ylabel('Feature')
    ax.set_yticks(range(len(FEATURE_COLUMNS)))
    ax.set_yticklabels(FEATURE_COLUMNS)
    ax.set_title(title)
    plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.show()

In [ ]:
# Temporal importance analysis
saliency = saliency_maps['gradient_x_input']
temporal_imp = aggregate_temporal_importance(saliency)[0].numpy()

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(range(SEQUENCE_LENGTH), temporal_imp, color='steelblue', alpha=0.7)
ax.set_xlabel('Days Before Prediction')
ax.set_ylabel('Importance')
ax.set_title('Temporal Importance: Which Days Matter Most?')
ax.set_xticks(range(0, SEQUENCE_LENGTH, 5))
ax.set_xticklabels([f'-{SEQUENCE_LENGTH - i}' for i in range(0, SEQUENCE_LENGTH, 5)])
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance analysis
feature_imp = aggregate_feature_importance(saliency)[0].numpy()

# Sort by importance
sorted_idx = np.argsort(feature_imp)[::-1]

fig, ax = plt.subplots(figsize=(10, 8))
colors = plt.cm.viridis(np.linspace(0, 1, len(FEATURE_COLUMNS)))
ax.barh(range(len(FEATURE_COLUMNS)), feature_imp[sorted_idx], color=colors)
ax.set_yticks(range(len(FEATURE_COLUMNS)))
ax.set_yticklabels([FEATURE_COLUMNS[i] for i in sorted_idx])
ax.set_xlabel('Importance')
ax.set_title('Feature Importance: Which Features Matter Most?')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Analyze saliency concentration across test samples
concentrations = []
predictions = []

model.eval()
with torch.no_grad():
    for features, labels in test_loader:
        saliency = saliency_computer.gradient_x_input(features)
        conc = compute_saliency_concentration(saliency).numpy()
        concentrations.extend(conc)
        
        preds = model(features).numpy().flatten()
        predictions.extend(preds)

concentrations = np.array(concentrations)
predictions = np.array(predictions)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of concentration
axes[0].hist(concentrations, bins=30, color='steelblue', alpha=0.7)
axes[0].axvline(concentrations.mean(), color='red', linestyle='--', label=f'Mean: {concentrations.mean():.3f}')
axes[0].set_xlabel('Saliency Concentration')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Saliency Concentration')
axes[0].legend()

# Concentration vs prediction confidence
confidence = np.abs(predictions - 0.5) * 2  # Scale to [0, 1]
axes[1].scatter(confidence, concentrations, alpha=0.5)
axes[1].set_xlabel('Prediction Confidence')
axes[1].set_ylabel('Saliency Concentration')
axes[1].set_title('Concentration vs Confidence')

plt.tight_layout()
plt.show()

## 6. Saliency-Based Trading Strategy

We'll create trading strategies that use saliency information.

In [ ]:
# Define interpretable features (features we expect to be important)
INTERPRETABLE_FEATURES = [
    FEATURE_COLUMNS.index('Close'),
    FEATURE_COLUMNS.index('Returns'),
    FEATURE_COLUMNS.index('RSI'),
    FEATURE_COLUMNS.index('MACD'),
    FEATURE_COLUMNS.index('Volatility')
]

print(f"Interpretable features: {[FEATURE_COLUMNS[i] for i in INTERPRETABLE_FEATURES]}")

In [ ]:
# Create different strategies
strategies = {
    'Baseline (No Filter)': SaliencyTradingStrategy(
        model,
        min_confidence=0.55,
        min_concentration=0.0,  # No concentration filter
        interpretable_features=None,  # No interpretability filter
        position_sizing='fixed'
    ),
    'Concentration Filter': SaliencyTradingStrategy(
        model,
        min_confidence=0.55,
        min_concentration=0.3,  # Require focused attention
        interpretable_features=None,
        position_sizing='fixed'
    ),
    'Interpretability Filter': SaliencyTradingStrategy(
        model,
        min_confidence=0.55,
        min_concentration=0.0,
        interpretable_features=INTERPRETABLE_FEATURES,
        position_sizing='fixed'
    ),
    'Full Saliency Filter': SaliencyTradingStrategy(
        model,
        min_confidence=0.55,
        min_concentration=0.3,
        interpretable_features=INTERPRETABLE_FEATURES,
        position_sizing='concentration'
    )
}

## 7. Backtesting and Comparison

In [ ]:
# Prepare test data for backtesting
# We need aligned prices and dates
df_clean = df_features.dropna()
test_start_idx = len(df_clean) - len(test_loader.dataset)
test_prices = df_clean['Close'].iloc[test_start_idx + SEQUENCE_LENGTH:]
test_dates = test_prices.index

# Get test features as numpy array
X_test = np.array([features.numpy() for features, _ in test_loader.dataset])

# Ensure alignment
min_len = min(len(X_test), len(test_prices))
X_test = X_test[:min_len]
test_prices = test_prices[:min_len]
test_dates = test_dates[:min_len]

print(f"Backtest samples: {len(X_test)}")
print(f"Date range: {test_dates[0]} to {test_dates[-1]}")

In [ ]:
# Run backtests
comparison = compare_strategies(
    strategies,
    X_test,
    test_prices,
    test_dates,
    initial_capital=100000,
    transaction_cost=0.001
)

print("Strategy Comparison:")
print(comparison)

In [ ]:
# Plot equity curves
fig, ax = plt.subplots(figsize=(14, 6))

results = {}
for name, strategy in strategies.items():
    result = run_backtest(strategy, X_test, test_prices, test_dates)
    results[name] = result
    ax.plot(result.equity_curve.index, result.equity_curve.values, label=name)

ax.set_xlabel('Date')
ax.set_ylabel('Equity ($)')
ax.set_title('Equity Curves: Saliency-Based Strategy Comparison')
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Analyze trade statistics for best strategy
best_strategy = 'Full Saliency Filter'
best_result = results[best_strategy]

print(f"\n{best_strategy} - Trade Analysis")
print(f"Total trades: {best_result.num_trades}")

if best_result.num_trades > 0:
    pnls = [t.pnl for t in best_result.trades]
    winning = [p for p in pnls if p > 0]
    losing = [p for p in pnls if p <= 0]
    
    print(f"Winning trades: {len(winning)}")
    print(f"Losing trades: {len(losing)}")
    print(f"Average win: ${np.mean(winning):.2f}" if winning else "No winning trades")
    print(f"Average loss: ${np.mean(losing):.2f}" if losing else "No losing trades")
    print(f"Largest win: ${max(pnls):.2f}")
    print(f"Largest loss: ${min(pnls):.2f}")

In [ ]:
# Plot drawdown comparison
fig, ax = plt.subplots(figsize=(14, 6))

for name, result in results.items():
    equity = result.equity_curve
    rolling_max = equity.expanding().max()
    drawdown = (equity - rolling_max) / rolling_max * 100
    ax.fill_between(drawdown.index, drawdown.values, alpha=0.3, label=name)

ax.set_xlabel('Date')
ax.set_ylabel('Drawdown (%)')
ax.set_title('Drawdown Comparison')
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Summary

In this notebook, we demonstrated how saliency maps can be used to:

1. **Understand model behavior**: Visualize which features and time steps the model focuses on
2. **Filter trading signals**: Only trade when the model's attention aligns with domain knowledge
3. **Adjust position sizes**: Size positions based on the concentration of model attention
4. **Improve risk management**: Avoid trades where the model focuses on potentially spurious patterns

Key findings:
- Saliency-filtered strategies tend to make fewer but higher-quality trades
- Concentration filtering helps identify high-conviction signals
- Interpretability filtering ensures the model's reasoning aligns with financial intuition